# Fine Tuning Practice
Mini Project for test

1. Gethering DATA, Load Dataset 

2. Tokenization 

3. Model Setup 

4. Training 

5. Inference 

In [2]:
from datasets import load_dataset
raw_datasets = load_dataset('imdb')

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 277386.38 examples/s]


In [5]:
from transformers import AutoTokenizer 

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

tokenized_datasets.set_format("torch")

Map: 100%|██████████| 50000/50000 [00:21<00:00, 2335.32 examples/s]


In [8]:
import torch 
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import DataLoader

In [9]:
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1_000))
train_dataset.set_format(type="torch", columns=['input_ids', 'attention_mask', 'label'])

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2)
model.to("cuda")

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for batch in train_dataloader:
    optimizer.zero_grad()

    inputs = {k: v.to("cuda") for k, v in batch.items()}

    # Forward pass 
    outputs = model(**inputs)
    loss = outputs.loss

    # Backward pass 
    loss.backward()

    optimizer.step()

